# TechMind · Fase 1 — Ingesta PDF, corpus y EDA

**Proyecto:** TechMind AI · Team 46 · Hackathon ONE G9  
**Archivo de trabajo:** `Fase1EDATechmindCOPIA.ipynb`  
**Kernel:** Python (TechMind EDA) → `C:\Team46\G9-LATAM-Team-46\.venv-eda\Scripts\python.exe`

Pipeline en cinco pasos: entorno y vault → contrato/mapeo → extracción → CSV filtrado → EDA (general vs train).

| Paso | Qué hace | Artefacto principal |
|:----:|----------|---------------------|
| 1 | Fija rutas y lista PDF | `data/raw/pdfs/` |
| 2 | Define schema y categorías | `MAPEO_CORPUS` |
| 3 | Extrae texto por página | `corpus_datos` |
| 4 | Guarda general + train | CSV en `data/` |
| 5 | Compara y visualiza | gráficos + métricas |


## 1. Entorno y vault de PDF

Se ancla el notebook a la raíz del repo (`C:\Team46\G9-LATAM-Team-46`) para no crear carpetas `data/` dentro de `notebooks/` cuando cambia el *cwd*. El vault canónico de binarios es `data/raw/pdfs/`. Si hay PDF en `dataset/raw/pdfs/`, se copian al vault. Sin al menos un PDF, el resto del pipeline no tiene sentido.


In [ ]:
# --- Paso 1: entorno y vault ---
%pip install -q "pymupdf>=1.24.0,<1.25.0" "pydantic>=2.0.0,<3.0.0" "pandas>=2.2.0,<2.3.0"

import os
import shutil
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from typing import List

import fitz  # PyMuPDF
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

PROJECT_ROOT = Path(r"C:\Team46\G9-LATAM-Team-46")
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"No existe PROJECT_ROOT: {PROJECT_ROOT}")

REPO_ROOT = PROJECT_ROOT
os.chdir(PROJECT_ROOT)

PATH_RAW_PDFS = (PROJECT_ROOT / "data" / "raw" / "pdfs").resolve()
PATH_RAW_CSV = (PROJECT_ROOT / "data" / "raw" / "csv").resolve()
PATH_PROCESSED = (PROJECT_ROOT / "data" / "processed").resolve()
PATH_ARCHIVE = (PROJECT_ROOT / "data" / "archive").resolve()
PATH_MODEL_ART = (PROJECT_ROOT / "model" / "artifacts").resolve()
PATH_LEGACY_PDFS = (PROJECT_ROOT / "dataset" / "raw" / "pdfs").resolve()

for d in [PATH_RAW_PDFS, PATH_RAW_CSV, PATH_PROCESSED, PATH_ARCHIVE, PATH_MODEL_ART]:
    d.mkdir(parents=True, exist_ok=True)

print("[ROOT]", PROJECT_ROOT)
print("[VAULT]", PATH_RAW_PDFS)

if PATH_LEGACY_PDFS.is_dir():
    n = 0
    for pdf in PATH_LEGACY_PDFS.glob("*.pdf"):
        dest = PATH_RAW_PDFS / pdf.name
        if (not dest.exists()) or dest.stat().st_size != pdf.stat().st_size:
            shutil.copy2(pdf, dest)
            n += 1
    if n:
        print(f"[MIGRACION] {n} PDF -> vault")

pdfs = sorted(PATH_RAW_PDFS.glob("*.pdf"))
print(f"PDFs en vault: {len(pdfs)}")
for p in pdfs:
    print(f"  - {p.name}")

if not pdfs:
    raise FileNotFoundError(f"Vault vacio: {PATH_RAW_PDFS}")


**Salida esperada:** lista de PDF bajo `data/raw/pdfs/`. Variables: `PROJECT_ROOT`, `PATH_RAW_PDFS`, `PATH_RAW_CSV`, `PATH_PROCESSED`.


## 2. Contrato de datos y mapeo de categorías

`DocumentoFragmento` (Pydantic) fija las 8 columnas del corpus. `MAPEO_CORPUS` asigna cada PDF a una categoría L1. Se excluye el PDF de redes basado en imagen (poco o nada de texto extraíble). Solo se procesarán archivos presentes en el vault **y** en el mapeo.


In [ ]:
# --- Paso 2: contrato + mapeo ---

class DocumentoFragmento(BaseModel):
    id_fragmento: str = Field(...)
    titulo_origen: str = Field(...)
    categoria_l1: str = Field(...)
    pagina: int = Field(...)
    texto_crudo: str = Field(...)
    longitud_caracteres: int = Field(...)
    fecha_extraccion: str = Field(...)
    hash_texto: str = Field(...)


# Excluido: Conceptos basicos de redes.pdf (PDF imagen)
MAPEO_CORPUS = {
    "Módulo 1 y 2 Fundamentos de Hardware.pdf": "Hardware",
    "Guía Técnica de Oracle Virtual Machine.pdf": "Sistemas_Operativos",
    "Java_Fundamental.pdf": "Lenguajes_Programacion",
    "IntroducciónPython_AnálisisDatos.pdf": "Lenguajes_Programacion",
    "Tkinter_Fundamental.pdf": "Lenguajes_Programacion",
    "SQL_Fundamental.pdf": "Bases_de_Datos",
    "Optimizando FMs.pdf": "Inteligencia_Artificial",
    "VisualizaciónInterpretacióndeDatos.pdf": "Inteligencia_Artificial",
    "Limpieza y Optimización de Datos en Pandas.pdf": "Inteligencia_Artificial",
    "Fundamentos de Redes y TCP.pdf": "Redes_y_Comunicaciones",
    "Manual de Estrategia y Control.pdf": "Arquitectura",
}

PATH_RAW_PDFS = Path(PATH_RAW_PDFS).resolve()
print("[MAPEO] Vault:", PATH_RAW_PDFS)
assert PATH_RAW_PDFS.is_dir(), PATH_RAW_PDFS

archivos_locales = {p.name for p in PATH_RAW_PDFS.glob("*.pdf")}
presentes = set(MAPEO_CORPUS) & archivos_locales
faltantes = set(MAPEO_CORPUS) - archivos_locales
extras = archivos_locales - set(MAPEO_CORPUS)

print(f"Presentes mapeados: {len(presentes)} / {len(MAPEO_CORPUS)}")
for f in sorted(presentes):
    print(f"  [OK] {f} -> {MAPEO_CORPUS[f]}")
if faltantes:
    print("Faltantes:")
    for f in sorted(faltantes):
        print("  -", f)
if extras:
    print("En vault sin mapeo (no se extraen):")
    for f in sorted(extras):
        print("  -", f)


**Salida esperada:** conteo de PDF mapeados / faltantes / extras. Listo para extracción.


## 3. Extracción de texto (bloques espaciales)

Por cada página se leen bloques de texto de PyMuPDF, ordenados por posición $(y, x)$, descartando ~6% superior e inferior (encabezados/pies). Cada página con texto genera un fragmento con `id_fragmento`, categoría, longitud y hash SHA-256. Los PDF solo-imagen no aportan páginas útiles.


In [ ]:
# --- Paso 3: extraccion ---

def extraer_texto_por_bloques_espaciales(page, margin_ratio_top=0.06, margin_ratio_bottom=0.06) -> str:
    height = page.rect.height
    min_y = height * margin_ratio_top
    max_y = height * (1.0 - margin_ratio_bottom)
    blocks = page.get_text("blocks")
    blocks_ordenados = sorted(blocks, key=lambda b: (b[1], b[0]))
    textos = []
    for b in blocks_ordenados:
        if b[6] == 0 and b[1] >= min_y and b[3] <= max_y:
            t = b[4].strip()
            if t:
                textos.append(t)
    return "\n\n".join(textos)


def pipeline_extraccion_masiva(input_dir, mapping: dict) -> List[dict]:
    input_dir = Path(input_dir).resolve()
    registros: List[dict] = []
    errores = 0
    archivos = sorted({p.name for p in input_dir.glob("*.pdf")} & set(mapping.keys()))
    print("EXTRACCION desde:", input_dir)
    print("PDFs a procesar:", len(archivos))
    if not archivos:
        print("[PELIGRO] Ningun PDF del mapeo en el vault.")
        return registros

    for pdf_name in archivos:
        categoria = mapping[pdf_name]
        pdf_path = input_dir / pdf_name
        try:
            doc = fitz.open(pdf_path)
            n_ok = 0
            h6 = hashlib.sha256(pdf_name.encode("utf-8")).hexdigest()[:6]
            pref = pdf_name.replace(".pdf", "").replace(" ", "_")[:12]
            for page_idx, page in enumerate(doc):
                texto = extraer_texto_por_bloques_espaciales(page)
                if not texto:
                    continue
                payload = {
                    "id_fragmento": f"{categoria}_{pref}_{h6}_p{page_idx + 1:03d}",
                    "titulo_origen": pdf_name,
                    "categoria_l1": categoria,
                    "pagina": page_idx + 1,
                    "texto_crudo": texto,
                    "longitud_caracteres": len(texto),
                    "fecha_extraccion": datetime.now(timezone.utc).isoformat(),
                    "hash_texto": hashlib.sha256(texto.encode("utf-8")).hexdigest(),
                }
                try:
                    registros.append(DocumentoFragmento(**payload).model_dump())
                    n_ok += 1
                except ValidationError:
                    errores += 1
            print(f"[OK] {pdf_name[:40]:<40} -> {n_ok:03d} paginas")
            doc.close()
        except Exception as e:
            print(f"[ERROR] {pdf_name}: {e}")
    print(f"Fragmentos: {len(registros)} | Fallos validacion: {errores}")
    return registros


PATH_RAW_PDFS = Path(PATH_RAW_PDFS).resolve()
print("[EXTRACT]", PATH_RAW_PDFS)
assert PATH_RAW_PDFS.is_dir()
assert list(PATH_RAW_PDFS.glob("*.pdf")), f"sin PDF en {PATH_RAW_PDFS}"
norm = str(PATH_RAW_PDFS).replace("\\", "/").lower()
assert norm.endswith("data/raw/pdfs"), f"ruta inesperada: {PATH_RAW_PDFS}"

corpus_datos = pipeline_extraccion_masiva(PATH_RAW_PDFS, MAPEO_CORPUS)
print("n_frag =", len(corpus_datos))


**Salida esperada:** `corpus_datos` (lista de dicts). Revisar páginas por PDF y total de fragmentos.


## 4. CSV general y dataset de entrenamiento

Se guardan dos archivos:

1. **General** — todo lo extraído (`data/raw/csv/dataset_general_v1.csv`).
2. **Entrenamiento** — solo fragmentos con longitud ≥ percentil 10 **y** ≥ 80 caracteres (`data/processed/dataset_entrenamiento_v1.csv`).

Así se separa la bóveda de auditoría del subconjunto usable para modelado.


In [ ]:
# --- Paso 4: persistencia + filtro ---

PATH_RAW_CSV = Path(PATH_RAW_CSV).resolve()
PATH_PROCESSED = Path(PATH_PROCESSED).resolve()
PATH_GENERAL_CSV = PATH_RAW_CSV / "dataset_general_v1.csv"
PATH_TRAIN_CSV = PATH_PROCESSED / "dataset_entrenamiento_v1.csv"

if not corpus_datos:
    print("Error: corpus vacio. Vault:", PATH_RAW_PDFS)
    df_maestro = pd.DataFrame()
    df_train = pd.DataFrame()
    umbral = None
else:
    df_maestro = pd.DataFrame(corpus_datos)
    PATH_RAW_CSV.mkdir(parents=True, exist_ok=True)
    PATH_PROCESSED.mkdir(parents=True, exist_ok=True)

    df_maestro.to_csv(PATH_GENERAL_CSV, index=False, encoding="utf-8")
    print("[PERSISTENCIA]", PATH_GENERAL_CSV)

    umbral = float(df_maestro["longitud_caracteres"].quantile(0.10))
    print(f"[ESTADISTICA] P10 = {umbral:.0f} chars")
    MIN_ABS = 80
    df_train = df_maestro[
        (df_maestro["longitud_caracteres"] >= umbral)
        & (df_maestro["longitud_caracteres"] >= MIN_ABS)
    ].copy().reset_index(drop=True)

    df_train.to_csv(PATH_TRAIN_CSV, index=False, encoding="utf-8")
    print("[PERSISTENCIA]", PATH_TRAIN_CSV)
    print("Retenidos:", len(df_train), "| Descartados:", len(df_maestro) - len(df_train))
    print(df_train["categoria_l1"].value_counts())


**Salida esperada:** dos CSV en `data/`. Variables `df_maestro`, `df_train`, `umbral`, `PATH_GENERAL_CSV`, `PATH_TRAIN_CSV`.


## 5. EDA: general vs entrenamiento

Carga ambos CSV (por si se reabre el notebook sin re-ejecutar todo), compara retención por categoría y muestra:

- tablas y muestra de texto
- impacto del filtro (general → train)
- gráficos de volumetría, longitud y balance de clases
- señales simples (ratio max/min, duplicados)


In [ ]:
# --- Paso 5: EDA dual (general + train) ---
import textwrap
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

_ROOT = Path(r"C:\Team46\G9-LATAM-Team-46")
PATH_GENERAL_CSV = Path(
    PATH_GENERAL_CSV
    if "PATH_GENERAL_CSV" in globals()
    else _ROOT / "data" / "raw" / "csv" / "dataset_general_v1.csv"
).resolve()
PATH_TRAIN_CSV = Path(
    PATH_TRAIN_CSV
    if "PATH_TRAIN_CSV" in globals()
    else _ROOT / "data" / "processed" / "dataset_entrenamiento_v1.csv"
).resolve()

if not PATH_GENERAL_CSV.is_file():
    raise FileNotFoundError(f"Falta CSV general:\n  {PATH_GENERAL_CSV}\nEjecuta el paso 4.")
if not PATH_TRAIN_CSV.is_file():
    raise FileNotFoundError(f"Falta CSV train:\n  {PATH_TRAIN_CSV}\nEjecuta el paso 4.")

df_general = pd.read_csv(PATH_GENERAL_CSV)
df_train = pd.read_csv(PATH_TRAIN_CSV)
df_maestro = df_general

print("[GENERAL]", PATH_GENERAL_CSV, "shape=", df_general.shape)
print("[TRAIN]  ", PATH_TRAIN_CSV, "shape=", df_train.shape)
print("Filtrados:", len(df_general) - len(df_train))
print(f"Retencion: {100 * len(df_train) / max(len(df_general), 1):.1f}%")

if "umbral" not in globals() or umbral is None:
    umbral = float(df_general["longitud_caracteres"].quantile(0.10))
    print(f"umbral P10 (recalculado): {umbral:.0f}")
else:
    print(f"umbral P10 (paso 4): {float(umbral):.0f}")

print("\n--- GENERAL (head) ---")
display(df_general.head(3))
print("\n--- TRAIN (head) ---")
display(df_train.head(5))

print("\n--- Muestra texto (train) ---")
_t = str(df_train.iloc[0]["texto_crudo"])
print("ID:", df_train.iloc[0]["id_fragmento"])
print(textwrap.fill(_t[:900] + ("..." if len(_t) > 900 else ""), width=85))

vc_g = df_general["categoria_l1"].value_counts().rename("general")
vc_t = df_train["categoria_l1"].value_counts().rename("train")
impacto = (
    pd.concat([vc_g, vc_t], axis=1)
    .fillna(0)
    .astype(int)
    .assign(
        descartados=lambda d: d["general"] - d["train"],
        retencion_pct=lambda d: (100 * d["train"] / d["general"].clip(lower=1)).round(1),
    )
    .sort_values("general", ascending=False)
)
print("\n--- Impacto del filtro por categoria ---")
display(impacto)

print("\n--- Longitud general ---")
print(df_general["longitud_caracteres"].describe().round(1))
print("\n--- Longitud train ---")
print(df_train["longitud_caracteres"].describe().round(1))

resumen_train = (
    df_train.groupby("categoria_l1")["longitud_caracteres"]
    .agg(count="count", media="mean", mediana="median", minimo="min", maximo="max")
    .round(1)
    .sort_values("count", ascending=False)
)
print("\n--- Longitud por categoria (train) ---")
display(resumen_train)

cmp = impacto.reset_index(names="categoria_l1")
order = cmp.sort_values("general", ascending=True)["categoria_l1"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
sns.barplot(data=cmp, y="categoria_l1", x="general", order=order, ax=axes[0], color="steelblue")
axes[0].set_title("General (crudo)")
axes[0].set_xlabel("Fragmentos")
axes[0].set_ylabel("")
sns.barplot(data=cmp, y="categoria_l1", x="train", order=order, ax=axes[1], color="seagreen")
axes[1].set_title("Entrenamiento (filtrado)")
axes[1].set_xlabel("Fragmentos")
axes[1].set_ylabel("")
fig.suptitle("Volumetria por categoria — general vs train", fontweight="bold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 4.5))
sns.histplot(df_general["longitud_caracteres"], bins=40, kde=True, color="steelblue", label="general", alpha=0.45)
sns.histplot(df_train["longitud_caracteres"], bins=40, kde=True, color="seagreen", label="train", alpha=0.55)
plt.axvline(float(umbral), color="crimson", ls="--", lw=1.5, label=f"P10≈{float(umbral):.0f}")
plt.axvline(80, color="darkorange", ls=":", lw=1.5, label="min 80")
plt.title("Distribucion de longitud: general vs train")
plt.xlabel("Caracteres")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5.5))
order_len = df_train.groupby("categoria_l1")["longitud_caracteres"].median().sort_values(ascending=False).index
sns.boxplot(
    data=df_train, y="categoria_l1", x="longitud_caracteres",
    order=order_len, palette="Set2", hue="categoria_l1", legend=False,
)
plt.axvline(float(umbral), color="crimson", ls="--", lw=1.2, label=f"P10≈{float(umbral):.0f}")
plt.title("Longitud por categoria (train)")
plt.xlabel("Caracteres")
plt.ylabel("")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
order_c = df_train["categoria_l1"].value_counts().index
ax = sns.countplot(
    data=df_train, y="categoria_l1", order=order_c,
    palette="Blues_r", hue="categoria_l1", legend=False,
)
plt.title("Balance de clases (train)")
plt.xlabel("Fragmentos")
plt.ylabel("")
for p in ax.patches:
    w = p.get_width()
    ax.annotate(
        f"{int(w)}", (w, p.get_y() + p.get_height() / 2.0),
        ha="left", va="center", xytext=(6, 0), textcoords="offset points",
        fontsize=10, fontweight="bold",
    )
plt.tight_layout()
plt.show()

vc = df_train["categoria_l1"].value_counts()
ratio = float(vc.max() / max(vc.min(), 1))
dup = int(df_train["texto_crudo"].duplicated().sum())
print("\n=== Senales de calidad (train) ===")
print(f"n_train={len(df_train)}  n_general={len(df_general)}")
print(f"categorias={df_train['categoria_l1'].nunique()}")
print(f"ratio max/min clase={ratio:.2f}  (>5 => class_weight o rebalance)")
print(f"duplicados texto={dup}")
print(f"mediana len train={df_train['longitud_caracteres'].median():.0f}")
print(f"mediana len general={df_general['longitud_caracteres'].median():.0f}")
print("Variables: df_general, df_train, df_maestro, impacto")


**Interpretación rápida:** si el ratio max/min de clases es alto, conviene `class_weight` o aumentar datos (`data/augmented/`). Si hay muchos duplicados, deduplicar antes de entrenar.
